# Multi-Disease Risk Analysis (Diabetes, Heart, Kidney)

This notebook extends your project from one disease to multiple diseases.

Diseases covered:
- Diabetes risk
- Heart disease risk
- Chronic kidney disease risk

Clinical caution: This is for screening support only, not diagnosis.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')

In [ ]:
# Dataset URL candidates (first working URL is used per disease)
DATASET_CANDIDATES = {
    'diabetes': [
        'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv'
    ],
    'heart': [
        'https://raw.githubusercontent.com/karanvyas22/Heart-Disease-Prediction/master/heart.csv',
        'https://raw.githubusercontent.com/je-suis-tm/machine-learning/master/datasets/heart.csv'
    ],
    'kidney': [
        'https://raw.githubusercontent.com/chaitanyakck/Chronic-Kidney-Disease-prediction/master/kidney_disease.csv',
        'https://raw.githubusercontent.com/siddhardhan23/Chronic-Kidney-Disease-Prediction/main/kidney_disease.csv'
    ]
}

def load_first_available(urls):
    for u in urls:
        try:
            d = pd.read_csv(u)
            return d, u
        except Exception:
            pass
    return None, None

loaded = {}
for disease, urls in DATASET_CANDIDATES.items():
    d, u = load_first_available(urls)
    loaded[disease] = {'df': d, 'url': u}
    if d is None:
        print(f'[WARN] Could not load {disease} dataset from candidates.')
    else:
        print(f'[OK] {disease}: {u} -> shape={d.shape}')

In [ ]:
def clean_kidney_dataframe(df):
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    for c in out.columns:
        if out[c].dtype == object:
            out[c] = out[c].astype(str).str.strip()

    target_candidates = ['classification', 'class', 'target', 'Outcome']
    target_col = None
    for t in target_candidates:
        if t in out.columns:
            target_col = t
            break
    if target_col is None:
        return None, None

    y = out[target_col].astype(str).str.lower()
    y = y.replace({'ckd': 1, 'notckd': 0, 'yes': 1, 'no': 0, '1': 1, '0': 0})
    y = pd.to_numeric(y, errors='coerce')

    X = out.drop(columns=[target_col])
    keep_cols = []
    for c in X.columns:
        xc = pd.to_numeric(X[c], errors='coerce')
        if xc.notna().mean() >= 0.4:
            X[c] = xc
            keep_cols.append(c)
    X = X[keep_cols]

    mask = y.notna()
    X = X.loc[mask].copy()
    y = y.loc[mask].astype(int).copy()
    return X, y

def run_binary_risk_pipeline(X, y, disease_name):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    num_cols = X.columns.tolist()
    preprocess = ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols)
    ])

    models = {
        'logistic_regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
        'random_forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight='balanced', random_state=RANDOM_STATE)
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    best_name, best_auc = None, -1
    for n, m in models.items():
        p = Pipeline([('preprocess', preprocess), ('model', m)])
        sc = cross_validate(p, X_train, y_train, cv=cv, scoring=['roc_auc'])
        mean_auc = sc['test_roc_auc'].mean()
        if mean_auc > best_auc:
            best_auc, best_name = mean_auc, n

    base = Pipeline([('preprocess', preprocess), ('model', models[best_name])])
    calibrated = CalibratedClassifierCV(estimator=base, cv=5, method='sigmoid')
    calibrated.fit(X_train, y_train)

    proba = calibrated.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.50).astype(int)

    metrics = {
        'disease': disease_name,
        'model': best_name,
        'roc_auc': roc_auc_score(y_test, proba),
        'pr_auc': average_precision_score(y_test, proba),
        'f1': f1_score(y_test, pred),
        'precision': precision_score(y_test, pred),
        'recall': recall_score(y_test, pred)
    }

    perm = permutation_importance(calibrated, X_test, y_test, scoring='roc_auc', n_repeats=10, random_state=RANDOM_STATE)
    imp = pd.DataFrame({'feature': X.columns, 'importance': perm.importances_mean}).sort_values('importance', ascending=False)
    return metrics, imp, X_test, y_test, proba


In [ ]:
results = []
importances = {}

# Diabetes setup
if loaded['diabetes']['df'] is not None:
    d = loaded['diabetes']['df'].copy()
    target = 'Outcome'
    invalid_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    for c in invalid_zeros:
        if c in d.columns:
            d[c] = d[c].replace(0, np.nan)
    X = d.drop(columns=[target])
    y = d[target]
    m, imp, _, _, _ = run_binary_risk_pipeline(X, y, 'diabetes')
    results.append(m)
    importances['diabetes'] = imp

# Heart setup
if loaded['heart']['df'] is not None:
    h = loaded['heart']['df'].copy()
    h.columns = [c.strip() for c in h.columns]
    heart_target = 'target' if 'target' in h.columns else ('output' if 'output' in h.columns else None)
    if heart_target is not None:
        X = h.drop(columns=[heart_target])
        y = h[heart_target]
        m, imp, _, _, _ = run_binary_risk_pipeline(X, y, 'heart')
        results.append(m)
        importances['heart'] = imp
    else:
        print('[WARN] Heart target column not found.')

# Kidney setup
if loaded['kidney']['df'] is not None:
    Xk, yk = clean_kidney_dataframe(loaded['kidney']['df'])
    if Xk is not None and len(Xk.columns) > 0:
        m, imp, _, _, _ = run_binary_risk_pipeline(Xk, yk, 'kidney')
        results.append(m)
        importances['kidney'] = imp
    else:
        print('[WARN] Kidney dataset preprocessing failed.')

results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
if len(results_df) > 0:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=results_df, x='disease', y='roc_auc')
    plt.title('ROC-AUC Comparison Across Diseases')
    plt.ylim(0.4, 1.0)
    plt.show()

for disease_name, imp in importances.items():
    print(f'\nTop features for {disease_name}:')
    display(imp.head(8))

## How to interpret this notebook
- Compare disease-specific metrics in `results_df`.
- Use top features to understand risk drivers for each disease.
- Use disease-specific thresholds in production (do not reuse one threshold for all diseases).
- Validate each disease model on external cohorts before real-world use.